<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Solution: Watch the system

**[Watch the system](https://learning.nextia-ai.com/courses/speech/m09/watch-the-system/)** · Speech AI: Transcribe, Understand and Speak · Module 9, lesson 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** score a recogniser call by call, compute the WER by slice, measure how much a weekly review sample moves by chance, and tell a real change in the calls from noise.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells. No account, no key and no model download. |
| **You should know** | WER by slice, from [The errors that matter](https://learning.nextia-ai.com/courses/speech/m02/the-errors-that-matter/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0 and matplotlib 3.11.2 (scikit-learn 1.9.1, jiwer 4.0.0 and faster-whisper 1.2.1 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m09/watch-the-system/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M09-L04-watch-the-system/watch-the-system-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile']:
    print(name, version(name))

## Setup: get the Larkfield calls

Run the next cell. It downloads the files that this lesson needs (about 2 MB) into a folder `larkfield-calls`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-calls` (in Colab: the **Files** panel on the left), then run the cell again.

Where the calls come from: Larkfield is an invented shop. Our code wrote the scripts, and open text-to-speech voices (Piper and Kokoro) read them. Every call is **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/data/dataset-card.md) names each voice and its licence. The calls are free to reuse under CC BY 4.0.

In [ ]:
# Setup: download the files into larkfield-calls/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
FILES = {
    "data/calls/calls.jsonl": "22fe4e147bf62b4ff6c3b6b4a1cc559b048e12a0c2b3c8098eb72982f2d9e695",
    "data/recordings/asr/whisper-small.jsonl": "953f41bdba51fe75c411960af3757a8c545b29670c87ac9f04b25f824ff3cf6d",
    "compare.py": "db394eb044e0423706644ba7c8e6451e4d21a268293b868032918ea2f30db85b",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-calls").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-calls and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-calls.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The question: how good are the transcripts this week?

A call centre's wallboard counts calls, waits and hang-ups for free. Listening to calls costs a person's time, so only a sample is checked. Larkfield's speech system works the same way: Grace can afford about 30 reviewed transcripts a week (60 minutes of review at about 2 minutes a call: the lesson's assumption).

To see what a sample can tell us, we need the result of every call. The course recorded Whisper small's transcripts of the 110 validation and test calls. The next cell scores each call against its reference, after the normalisation guide.

In [ ]:
calls = load_calls()
hyp = {}
for line in open("data/recordings/asr/whisper-small.jsonl", encoding="utf-8"):
    r = json.loads(line)
    if r["kind"] == "phone":
        hyp[r["call_id"]] = r["text"]

rows = []                                            # one row per call: its errors and its reference words
for cid, c in sorted(calls.items()):
    if c["split"] == "train" or cid not in hyp:
        continue
    k = cp.counts(cp.normalise(reference(c), "full", c["language"]), cp.normalise(hyp[cid], "full", c["language"]))
    rows.append({"call_id": cid, "line": c["line"], "language": c["language"],
                 "errors": k["sub"] + k["del"] + k["ins"], "words": k["ref_words"]})

def wer_of(rs):
    """Corpus WER: all errors divided by all reference words (not the mean of the calls' WERs)."""
    return sum(r["errors"] for r in rs) / max(sum(r["words"] for r in rs), 1)

print(len(rows), "calls; WER of all of them:", round(wer_of(rows), 4))

> **Check.** 110 calls, WER 7.9% on Larkfield's synthetic calls.

## 2. Your turn: the WER by slice

A **slice** is a group of calls that share one property, such as the phone line or the language. The total hides the slices.

> **Your turn.** Write `slice_wer(rows, key)`. For each value of `rows[i][key]`, in sorted order, return `(number of calls, WER rounded to 4 digits)`, with the WER computed by `wer_of()`.

In [ ]:
def slice_wer(rows, key):
    """{value: (number of calls, WER)} for each value of `key` ("line", "language", ...), WER as wer_of()."""
    out = {}
    for value in sorted({r[key] for r in rows}):
        part = [r for r in rows if r[key] == value]
        out[value] = (len(part), round(wer_of(part), 4))
    return out

In [ ]:
by_line = slice_wer(rows, "line")
by_language = slice_wer(rows, "language")
print("by line:    ", by_line)
print("by language:", by_language)

In [ ]:
expect_hash('by_line', by_line, '4938b78f0cf1fa66')

> **Check.** English 4.0% and French 14.1%: if a new region brings more French calls, the total rises although the recogniser did not change. The G.726 line has 9.3% on only 16 calls: a small slice.

## 3. How much does a weekly sample move?

Draw many samples of the same size from the 110 results, with replacement, as if each week brought calls like these. The spread of their WERs is the noise of a sample of that size.

> **Predict.** With 30 calls a week, how wide is the range that holds 90% of the weekly WERs?

In [ ]:
rng = np.random.default_rng(41)
sizes = [10, 20, 30, 50, 100]
spread = {}
for n in sizes:
    samples = [wer_of([rows[i] for i in rng.integers(0, len(rows), n)]) for _ in range(2000)]
    spread[n] = (np.percentile(samples, 5), np.median(samples), np.percentile(samples, 95))
    print(f"{n:3} calls: 90% of samples between {spread[n][0]:.1%} and {spread[n][2]:.1%} (median {spread[n][1]:.1%})")

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.fill_between(sizes, [spread[n][0] for n in sizes], [spread[n][2] for n in sizes], alpha=0.25, label="90% of samples")
ax.plot(sizes, [spread[n][1] for n in sizes], marker="o", label="median")
ax.set_xlabel("calls in the weekly review sample"); ax.set_ylabel("sample WER"); ax.legend(); plt.tight_layout(); plt.show()

> **Check.** 10 calls: 4.4% to 12.2%. 30 calls: 5.9% to 10.4%. 100 calls: 6.8% to 9.2%. The range narrows slowly: three times more review time buys only about half the width.

## 4. A new phone provider (a constructed example)

From week 5, a new provider carries 60% of the calls on the G.726 line, which has a higher WER. The next cell builds eight weeks of 30 reviewed calls from the real per-call results. It is a constructed example, not a measurement.

> **Predict.** Will the weekly WER show the change clearly from week 5?

In [ ]:
rng = np.random.default_rng(9)
voip = [r for r in rows if r["line"] == "voip_g726"]
rest = [r for r in rows if r["line"] != "voip_g726"]
share_now = len(voip) / len(rows)
weeks = []
for week in range(1, 9):
    share = share_now if week <= 4 else 0.6          # a new provider from week 5: 60% of calls on G.726
    n_voip = int(rng.binomial(30, share))
    pick = ([voip[i] for i in rng.integers(0, len(voip), n_voip)] +
            [rest[i] for i in rng.integers(0, len(rest), 30 - n_voip)])
    weeks.append((week, n_voip, wer_of(pick)))
    print(f"week {week}: {n_voip:2} of 30 calls on G.726, sample WER {wer_of(pick):.1%}")

low, high = spread[30][0], spread[30][2]
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.axhspan(low, high, alpha=0.15, label="normal range for 30 calls")
ax.plot([w for w, _, _ in weeks], [x for _, _, x in weeks], marker="o", label="weekly sample WER")
ax.axvline(4.5, color="black", lw=0.8, ls="--"); ax.set_xlabel("week (new provider from week 5)")
ax.legend(loc="upper left"); plt.tight_layout(); plt.show()

> **Check.** Weeks 1 to 4: 10.6%, 7.5%, 7.8%, 7.2%. Weeks 5 to 8: 7.4%, 10.5%, 8.6%, 8.5%. The change hides in the noise, and week 1 is above the range with no change at all. But count the calls on the G.726 line: 7, 3, 4, 5 of 30 before, 17, 18, 16, 18 after. **The input counts show the change at once**, for free and for every call. Then review more calls from the new slice.

## 5. Failure case: the mean of the calls' WERs

A common mistake is to average the WER of each call. Short calls then count as much as long ones.

In [ ]:
mean_of_rates = np.mean([r["errors"] / r["words"] for r in rows])
print(f"mean of the calls' WERs: {mean_of_rates:.2%}   corpus WER: {wer_of(rows):.2%}")
short = sorted(rows, key=lambda r: r["words"])[:3]
print("the three shortest calls:", [(r["call_id"], r["words"], round(r["errors"] / r["words"], 3)) for r in short])

> **Check.** The mean of the calls' WERs is 7.69%, against a corpus WER of 7.92%. Here they are close, because every synthetic call has 90 words or more. Real calls vary much more: a 10-word call with 3 errors counts as 30%, as much as a long call with the same rate. Always sum the errors and the words first, then divide, as the course's comparison table does.

## Next

You scored slices, measured the noise of a review sample, and saw why input counts find drift first. The [lesson page](https://learning.nextia-ai.com/courses/speech/m09/watch-the-system/) adds the IVR outcomes, new product names and captions, and has the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Choose a recogniser for two languages (case study)](https://learning.nextia-ai.com/courses/speech/m10/choose-a-recogniser-for-two-languages/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m09/watch-the-system/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every call is synthetic: open text-to-speech voices read scripts that we invented. No real customer and no real person's voice is in this data.